In [2]:
import numpy as np
import sys
import math
import time
import random
import json

In [3]:
#Funciones para guardar y cargar instancias y datos en archivos JSON
def save_instance_file(sat_instance_descriptor, file_path="BiKnapsack_instance.json"):
    json_str = json.dumps(sat_instance_descriptor)
    file_path 
    with open(file_path, 'w') as file:
        json.dump(json_str, file)
    return 0

def load_file_instance(file_path='BiKnapsack_instance.json'):
    file = open(file_path,)
    data = json.load(file)
    array = json.loads(data)
    return array

def readFile_json(filepath):
    file = open(filepath, "r")
    content = json.load(file)
    
    file.close()
    n=content['item_count']
    wmax=content['capacity']
    values=content['item_values']
    weights=content['item_weights']

    return n, wmax, values, weights    

In [1]:
#Función de generación de instancias -> instancia del problema de la mochila binivel, [tamaño1, tamaño2, items], cada item en items es [id, tamaño, valor]
def bilevel_knapsack_instance_generator(upper_knap_size, lower_knap_size, number_items):     
    biknap_instance_descriptor=[upper_knap_size, lower_knap_size]
    items=[]
    if(upper_knap_size<=lower_knap_size):
        knap_size=upper_knap_size
    else:
        knap_size=lower_knap_size
    max_item_size=int(knap_size/10)
    for i in range(number_items):
        items.append([i, random.randint(1, max_item_size), random.randint(1,500)])
    biknap_instance_descriptor.append(items)
    return biknap_instance_descriptor

def bilevel_knapsack_instance_instancer(items, biknap_instance_descriptor):
    for item in items:    #Comprobar que todos los objetos del conjunto pertenecen al problema
        if not(item in biknap_instance_descriptor[2]):
            print("Conjunto de items inválido: " + str(item) + " no pertenece al conjunto de objetos")
            return 0
    return 1

def random_x(n, t=1, p=0.5):   #genera t individuos aleatorios de tamaño n
    xs=[]
    for i in range(t):
        x=[]
        for i in range(n):
            #x.append(random.randint(0,1))
            a=random.random()
            if a>p:
                x.append(1)
            else:
                x.append(0)
        xs.append(x)
    return xs
        

#fitness del segundo nivel del algoritmo gnético (busca maximizar su beneficio, minimizar el espacio no usado de su mochila)
def fitness_check_knapsack(items, biknap_instance_descriptor):     
    sumatorio_peso=0    # si el fitness[0] es negativo, la solución es inválida, si es positivo, válida. Ambas son peores en su nivel de validez cuanto más grandes sean en valor absoluto
    sumatorio_valor=0   # el fitness[1] es el verdadero valor de medida de calidad y es válido solo si el fitness[0] es positivo
    for item in items:
        sumatorio_peso+=item[1]
        sumatorio_valor+=item[2]
    fitness= [biknap_instance_descriptor[1]-sumatorio_peso, sumatorio_valor]
    return fitness   
    
#fitness del primer nivel del algoritmo genético (busca maximizar el valor y minimizar el espacio no usado en la mochila del primer nivel, añadiendo también el fitness del mejor individuo del segundo nivel para una instancia del problema sin los items del primer nivel)
def fitness_check_upper_knapsack(items, best_lower_fitness, biknap_instance_descriptor): 
    sumatorio_peso=0    # si el fitness[0] es negativo, la solución es inválida, si es positivo, válida. Ambas son peores en su nivel de validez cuanto más grandes sean en valor absoluto
    sumatorio_valor=0   # el fitness[1] no es verdaderamente significativo, pero puede ser un último factor diferenciador, la solución es mejor cuanto mayor sea
    #el fitness[2] y fitness[3] indican la calidad de la mejor solución encontrada por el algoritmo de segundo nivel sin los items que escogió el individuo cuyo fitness se está evaluando. 
    #Son los verdaderos valores de medida de calidad, la solución es mejor si el fitness[2] es negativo (la mejor solución encontrada por el segundo nivel es inválida) o, en caso de que fitness[2] sea positivo, cuanto menor sea fitness[3] y mayor sea fitness[1]
    for item in items:
        sumatorio_peso+=item[1]
        sumatorio_valor+=item[2]
    fitness= [biknap_instance_descriptor[0]-sumatorio_peso, sumatorio_valor, best_lower_fitness[0], best_lower_fitness[1]]
    return fitness  

#función para eliminar del conjunto de items disponibles para el segundo nivel los escogidos por el primer nivel (o pone su valor a 0, según DNeg)
def adjust_allowed_items(items, biknap_instance_descriptor):
    lower_biknap_instance_descriptor=biknap_instance_descriptor.copy()
    for item in items:
        lower_biknap_instance_descriptor[2].remove(item)
    return lower_biknap_instance_descriptor

#función para arreglar la primera generación de individuos de manera voraz para que siempre tengan solución válida
def greedy_fix(items, biknap_instance_descriptor, lower=0):
    items_fixed=items.copy()
    sumatorio_peso=0
    max_peso_item=0
    item_eliminar=items_fixed[0]
    if(lower):
        max_peso_mochila=biknap_instance_descriptor[1]
    else:
         max_peso_mochila=biknap_instance_descriptor[0]
    
    for item in items_fixed:
        sumatorio_peso+=item[1]
        if item[1]>max_peso_item:
            max_peso_item=item[1]
            item_eliminar=item
    valid=max_peso_mochila-sumatorio_peso   
    
    while valid<0:
        items_fixed.remove(item_eliminar)
        sumatorio_peso=0
        max_peso_item=0
        item_eliminar=items_fixed[0]
        for item in items_fixed:
            sumatorio_peso+=item[1]
            if item[1]>max_peso_item:
                max_peso_item=item[1]
                item_eliminar=item
        valid=max_peso_mochila-sumatorio_peso  
        
    return items_fixed
            

In [5]:
#generacion de instancias en lotes de n_instnaces*len(hp[1]), donde hp es [n_items, tamaños_mochilas_casos] y tamaños_mochilas_casos es [[tamaño mochila 1_1, tamaño mochila 2_1], [tamaño mochila 1_2, tamaño mochila 2_2], etc.]
def instance_generator_batch(filename_generic, n_instances, hp):
    n_items=hp[0]
    resultados=[]
    for i in range(len(hp[1])):
        knap_size_1=hp[1][i][0]
        knap_size_2=hp[1][i][1]
        for j in range(n_instances):
            instance_filename=filename_generic+"_"+str(n_items)+"_"+str(knap_size_1)+"_"+str(j+1)+".json"
            instance=bilevel_knapsack_instance_generator(knap_size_1, knap_size_2, n_items)
            save_instance_file(instance, instance_filename)

            resultados.append(instance_filename)

    return resultados

In [6]:
#Función que recibe una solución de items del problema y devuelve un array de 1s y 0s que indica qué items del problema pertenecen a la solución
def encoder(items, biknap_instance_descriptor):   
    encoded_solution=[]
    items_in_problem=biknap_instance_descriptor[2]
    items_ids=[]
    for i in items:
        items_ids.append(i[0])
    for i in range(len(items_in_problem)):
        if (items_in_problem[i][0] in items_ids):
            encoded_solution.append(1)
        else:
            encoded_solution.append(0)
    return encoded_solution

#Función inversa al encoder, recibe un array de 1s y 0s y devuelve los items del problema que representan
def decoder(encoded_solution, biknap_instance_descriptor):
    decoded_solution=[]
    items_in_problem=biknap_instance_descriptor[2]
    for i in range(len(encoded_solution)):
        if encoded_solution[i]:
            decoded_solution.append(items_in_problem[i])
    return decoded_solution

In [3]:
#Función de cruce
def cruce_1P(population, prob_mut=0.01):   #función de cruce en un punto
    indices_pob=[]
    hijos=[]
   
    tamanho=len(population[0])
     
    for i in range(len(population)):
        indices_pob.append(i)
        
    for i in range(int((len(population))/2)):
        
        p1=random.choice(indices_pob)
        padre1= population[p1]
        indices_pob.remove(p1)
        p2=random.choice(indices_pob)
        padre2=population[p2]
        indices_pob.remove(p2)
        
        punto_corte=random.choice(list(range(tamanho-1))[1:])
        padre1_1=padre1[:punto_corte]
        padre1_2=padre1[punto_corte:]
        padre2_1=padre2[:punto_corte]
        padre2_2=padre2[punto_corte:]
        hijo1=padre1_1 + padre2_2
        hijo2=padre2_1 + padre1_2
        
        for i in range(len(hijo1)):  #Aplica la mutación a los hijos generados
            muta1=random.random()
            muta2=random.random()
            if muta1<=prob_mut:
                hijo1[i]=int(not hijo1[i])
            if muta2<=prob_mut:
                hijo2[i]=int(not hijo2[i])      
        
        hijos.append(hijo1)
        hijos.append(hijo2)
        
        #print(padre1, padre2, punto_corte)
        #print(hijo1, hijo2)
    
    return hijos

In [8]:
#funciones de selección
#priorizan el valor máximo y solo son válidas si el espacio restante es positivo
#en el primer nivel también se prioriza minimizar el espacio libre del segundo nivel manteniéndolo en positivo
#segundo nivel: fitness a es mejor que fitness b si, a[0] es positivo y < b[0] y a[1] >=b[1]
#primer nivel: fitness a es mejor que fitness b si, a[0] es positivo y < b[0] y (a[2] < 0 y a[2] < b[2]) o (a[2] > 0 y a[2] > b[2]) y a[3] < b[3]

#FUNCIONES DE SELECCIÓN PARA EL SEGUNDO NIVEL
def selection_elitism_lower_knap(y_pop, y_hijos, fit_list, knap_instance_descriptor, fix=False, fix_rate=0.5):   #función de selección elitista para el segundo nivel
    new_gen=[]
    new_fit_list=[]
    y_pop_sorted=[]
    tam_pop=len(y_pop)
    
    for y_hijo in y_hijos:    #para cada hijo se calcula su fitness
        y_hijo_items=decoder(y_hijo, knap_instance_descriptor)
        fit=fitness_check_knapsack(y_hijo_items, knap_instance_descriptor)
        if fix and random.random()<fix_rate:
            if fit[0]<0:
                y_hijo_items_fixed=greedy_fix(y_hijo_items, knap_instance_descriptor, 1)
                y_hijo_fixed=encoder(y_hijo_items_fixed, knap_instance_descriptor)
                fit=fitness_check_knapsack(y_hijo_items_fixed, knap_instance_descriptor)
                y_pop.append(y_hijo_fixed)
            else:
                y_pop.append(y_hijo)
        else:
            y_pop.append(y_hijo)
            
        fit_list.append(fit)

    fit_list_sorted=sort_fitness_lower_knap(fit_list)    #se ordenan los fitness de padres e hijos
    for fit_i in fit_list_sorted:
        y_pop_sorted.append(y_pop[fit_i[2]])     #se ordenan los individuios de acuredo al orden del fitness

    for i in range(len(y_pop_sorted)):
        y_pop_sorted[i]=[y_pop_sorted[i], [fit_list_sorted[i][0], fit_list_sorted[i][1]]]  
    
    for i in range(tam_pop):   #los individuos con el mayor fitness sobreviven
       new_gen.append(y_pop_sorted[i][0])
       new_fit_list.append(y_pop_sorted[i][1])
        
    return new_gen, new_fit_list

def sort_fitness_lower_knap(fit_list):     #Dada una lista de fitness del tipo [diferencial peso, valor], función que ordena los fitness para el segundo nivel
    fit_list_index=[]
    valid_fitness=[]
    invalid_fitness=[]
    
    for i in range(len(fit_list)):    #se añade el índice del orden original a la lista
        fit_list_index.append([fit_list[i][0], fit_list[i][1], i])
    
    for i in fit_list_index:    #se separan los fitness válidos de los inválidos (los inválidos tienen un peso mayor que la capacidad de la mochila, por lo que el diferencial es negativo)
        if i[0]>=0:
            valid_fitness.append(i)
        else:
            invalid_fitness.append(i)
            
    if not valid_fitness:    #si no hay fitness válidos se ordenan los inválidos
        sorted_fitness=list(reversed(sorted(invalid_fitness, key=lambda ind: ind[1])))
        sorted_fitness=list(sorted(sorted_fitness, key=lambda ind: abs(ind[0])))
    else:    #si los hay se ordenan los válidos
        sorted_fitness=list(reversed(sorted(valid_fitness, key=lambda ind: ind[1])))
        if invalid_fitness:   #y si hay inválidos se ordenan y añanaden a los válidos
            invalid_fitness=list(reversed(sorted(invalid_fitness, key=lambda ind: ind[1])))
            invalid_fitness=list(sorted(invalid_fitness, key=lambda ind: abs(ind[0])))
            sorted_fitness=sorted_fitness+invalid_fitness

    return sorted_fitness

#FUNCIONES DE SELECCIÓN PARA EL PRIMER NIVEL
def sort_fitness_lower_knap_upper_ver(fit_list):     #Dada una lista de fitness del tipo [diferencial peso, valor], función que ordena los fitness del segundo nivel en los individuos de primer nivel
    fit_list_index=[]
    valid_fitness=[]
    invalid_fitness=[]
    
    for i in range(len(fit_list)):    #se añade el índice del orden original a la lista
        fit_list_index.append([fit_list[i][0], fit_list[i][1], fit_list[i][-1]])
    
    for i in fit_list_index:    #se separan los fitness válidos de los inválidos (los inválidos tienen un peso mayor que la capacidad de la mochila, por lo que el diferencial es negativo)
        if i[0]>=0:
            valid_fitness.append(i)
        else:
            invalid_fitness.append(i)
            
    if not valid_fitness:    #si no hay fitness válidos se ordenan los inválidos
        sorted_fitness=list(reversed(sorted(invalid_fitness, key=lambda ind: ind[1])))
        sorted_fitness=list(sorted(sorted_fitness, key=lambda ind: abs(ind[0])))
    else:    #si los hay se ordenan los válidos
        sorted_fitness=list(reversed(sorted(valid_fitness, key=lambda ind: ind[1])))
        if invalid_fitness:   #y si hay inválidos se ordenan y añanaden a los válidos
            invalid_fitness=list(reversed(sorted(invalid_fitness, key=lambda ind: ind[1])))
            invalid_fitness=list(sorted(invalid_fitness, key=lambda ind: abs(ind[0])))
            sorted_fitness=sorted_fitness+invalid_fitness

    return sorted_fitness

#Función de selección elitista para individuos del primer nivel, calcula el fintess de los hijos llamando al algoritmo genético de segundo nivel
def selection_elitism_upper_knap(x_pop, x_hijos, fit_list, knap_instance_descriptor, file_path, swarmsize_lower=40, iterations_lower=50, zeros_initial_proportion=0.5, prob_mutation=0.01, it_fix_lower=10, fix=False, fix_rate=0.5, debug_check=False):   #función de selección elitista para el primer nivel
    new_gen=[]
    new_fit_list=[]
    x_pop_sorted=[]
    tam_pop=len(x_pop)
    
    for x_hijo in x_hijos:    #para cada hijo se calcula su fitness
        x_hijo_items=decoder(x_hijo, knap_instance_descriptor)
        lower_knap_instance_descriptor = adjust_allowed_items(x_hijo_items, load_file_instance(file_path))
        lower_knap_solution=Genetic_lower_knap(lower_knap_instance_descriptor, swarmsize_lower, iterations_lower, zeros_initial_proportion, prob_mutation, it_fix_lower)
        fitness=fitness_check_upper_knapsack(x_hijo_items, lower_knap_solution[1], knap_instance_descriptor)
        if fix:
            if fitness[0]<0 and random.random()<fix_rate:
                x_hijo_items_fixed=greedy_fix(x_hijo_items, knap_instance_descriptor)
                x_hijo_fixed=encoder(x_hijo_items_fixed, knap_instance_descriptor)
                lower_knap_instance_descriptor = adjust_allowed_items(x_hijo_items_fixed, load_file_instance(file_path))
                lower_knap_solution=Genetic_lower_knap(lower_knap_instance_descriptor, swarmsize_lower, iterations_lower, zeros_initial_proportion, prob_mutation, it_fix_lower)
                fitness=fitness_check_upper_knapsack(x_hijo_items_fixed, lower_knap_solution[1], knap_instance_descriptor)
                x_pop.append(x_hijo_fixed)
            else:
                x_pop.append(x_hijo)
        else:
            x_pop.append(x_hijo)
            
        fit_list.append(fitness)
    

    fit_list_sorted=sort_fitness_upper_knap(fit_list, debug_check)    #se ordenan los fitness de padres e hijos
    for fit_i in fit_list_sorted:
        x_pop_sorted.append(x_pop[fit_i[-1]])     #se ordenan los individuios de acuredo al orden del fitness

    for i in range(len(x_pop_sorted)):
        x_pop_sorted[i]=[x_pop_sorted[i], [fit_list_sorted[i][0], fit_list_sorted[i][1], fit_list_sorted[i][2],  fit_list_sorted[i][3]]]  
    
    if(debug_check):
        print("selection:", tam_pop, len(fit_list), len (fit_list_sorted), len(x_pop_sorted))
    
    for i in range(tam_pop):   #los individuos con el mayor fitness sobreviven
       new_gen.append(x_pop_sorted[i][0])
       new_fit_list.append(x_pop_sorted[i][1])
        
    return new_gen, new_fit_list


def sort_fitness_upper_knap(fit_list, debug_check=False):  #Dada una lista de fitness del tipo [diferencial peso en mochila superior, valor en la mochila superior, diferencial peso en mochila inferior, valor en la mochila inferior], función que ordena los fitness para el primer nivel
    fit_list_index=[]
    true_valid_fitness=[]
    true_invalid_fitness=[]
    lower_valid_fitness=[]
    lower_invalid_fitness=[]
    sorted_fitness=[]
    sorted_fitness_valid=[]
    sorted_fitness_invalid=[]
    
    for i in range(len(fit_list)):    #se añade el índice del orden original a la lista
        fit_list_index.append([fit_list[i][0], fit_list[i][1], fit_list[i][2], fit_list[i][3], i])
    
    for i in fit_list_index:    #se separan los fitness válidos de los inválidos (los inválidos tienen un peso mayor que la capacidad de la mochila, por lo que el diferencial es negativo)
        if i[0]>=0:
            true_valid_fitness.append(i)
        else:
            true_invalid_fitness.append(i)
            
    if true_valid_fitness:   #si hay fitness válidos se ordenan los válidos primero
        for i in true_valid_fitness:    
            lower_valid_fitness.append([i[2], i[3], i[-1]])
        #presorted_fitness_valid=list(reversed(sort_fitness_lower_knap(lower_valid_fitness)))
        presorted_fitness_valid=list(reversed(sort_fitness_lower_knap_upper_ver(lower_valid_fitness)))
        for i in range(len(presorted_fitness_valid)):
            for j in range(len(true_valid_fitness)):
                if(true_valid_fitness[j][-1] == presorted_fitness_valid[i][-1]):
                    sorted_fitness_valid.append([true_valid_fitness[j][0], true_valid_fitness[j][1], presorted_fitness_valid[i][0], presorted_fitness_valid[i][1], presorted_fitness_valid[i][-1]])
            
    if true_invalid_fitness:     #y si hay inválidos se ordenan por separado y añanaden después de los válidos
        for i in true_invalid_fitness:
            lower_invalid_fitness.append([i[2], i[3], i[-1]])
        #presorted_fitness_invalid=list(reversed(sort_fitness_lower_knap(lower_invalid_fitness)))
        presorted_fitness_invalid=list(reversed(sort_fitness_lower_knap_upper_ver(lower_invalid_fitness)))
        for i in range(len(presorted_fitness_invalid)):
            for j in range(len(true_invalid_fitness)):
                if(true_invalid_fitness[j][-1] == presorted_fitness_invalid[i][-1]):
                    sorted_fitness_invalid.append([true_invalid_fitness[j][0], true_invalid_fitness[j][1], presorted_fitness_invalid[i][0], presorted_fitness_invalid[i][1], presorted_fitness_invalid[i][-1]])
    
    sorted_fitness=sorted_fitness_valid+sorted_fitness_invalid
    
    if(debug_check):
        print("sort:", len(true_valid_fitness), len(lower_valid_fitness), len(presorted_fitness_valid), "|", len(true_invalid_fitness), len(lower_invalid_fitness), len(presorted_fitness_invalid), "|", len(sorted_fitness_valid), len(sorted_fitness_invalid), len(sorted_fitness)) 
        indexes_sorted_fitness=[]
        for i in range (len(sorted_fitness)):
            indexes_sorted_fitness.append(sorted_fitness[i][-1]) 
        print("final:", len(indexes_sorted_fitness), indexes_sorted_fitness) 
        valid_sorted_fitness=[]
        for i in range (len(true_valid_fitness)):
            valid_sorted_fitness.append(true_valid_fitness[i][-1]) 
        print("valid:", len(valid_sorted_fitness),valid_sorted_fitness) 
        invalid_sorted_fitness=[]
        for i in range (len(true_invalid_fitness)):
            invalid_sorted_fitness.append(true_invalid_fitness[i][-1]) 
        print("invalid:", len(invalid_sorted_fitness), invalid_sorted_fitness)

    return sorted_fitness

In [5]:
#algoritmo genético (se arregla la generación entera vorazmente cada it_fix iteraciones y un fix_rate>=1, se arregla cada individuo vorazmente con una probabilidad de fix_rate y un it_fix de 1, y se arreglan todos los individuos inválidos que se generen con un it_fix de 1 y fix_rate>=1)
#Algoritmo genético inferior
def Genetic_lower_knap(knap_instance_descriptor, swarmsize=40, iterations=50, zeros_initial_proportion=0.5, prob_mutation=0.01, it_fix=10, fix_rate=0.5):
    fit_list=[]
    first_gen_fixed=[]
    best_inds=[]
    best_fits=[]

    random_enc_pop=random_x(len(knap_instance_descriptor[2]), swarmsize, zeros_initial_proportion)
    
    for ind in random_enc_pop:
        fit=fitness_check_knapsack(decoder(ind, knap_instance_descriptor), knap_instance_descriptor)
        if fit[0]<0 and random.random()<fix_rate:
            fixed_ind_items=greedy_fix(decoder(ind, knap_instance_descriptor), knap_instance_descriptor, 1)
            fixed_ind=encoder(fixed_ind_items,knap_instance_descriptor) 
            fit=fitness_check_knapsack(fixed_ind_items, knap_instance_descriptor)
            first_gen_fixed.append(fixed_ind)
        else:
            first_gen_fixed.append(ind)
        fit_list.append(fit)
        
        

    new_gen=first_gen_fixed.copy()
    new_fit_list=fit_list.copy()
    
    for it in range(iterations):
        fix= it%it_fix==0
        hijos=cruce_1P(new_gen, prob_mutation)
        new_gen, new_fit_list=selection_elitism_lower_knap(new_gen, hijos, new_fit_list, knap_instance_descriptor, fix, fix_rate)
        
        best_inds.append(new_gen[0])
        best_fits.append(new_fit_list[0])
    
    best_ind=best_inds[-1]
    best_fit=best_fits[-1]
    
    return best_ind, best_fit, best_inds, best_fits

#GENÉTICO SUPERIOR
#generacion de individuos (creación padres originales/cruce)
#eliminar/poner a 0 los items del individuo
#fitness -> Algoritmo genético -> mejor valor válido del segundo nivel
#selección en base al fitness
#repetir
def Genetic_bilevelKnapsack(file_path, swarmsize=40, iterations=50, swarmsize_lower=40, iterations_lower=50, zeros_initial_proportion=0.5, prob_mutation=0.01, it_fix=10, it_fix_lower=10, fix_rate=0.5):
    
    fit_list=[]
    first_gen_fixed=[]
    lower_ind_list=[]
    best_inds=[]
    best_fits=[]

    knap_instance_descriptor=load_file_instance(file_path)
    random_enc_pop=random_x(len(knap_instance_descriptor[2]), swarmsize, zeros_initial_proportion) 
    
    for ind in random_enc_pop:
        pre_fit=fitness_check_knapsack(decoder(ind, knap_instance_descriptor), knap_instance_descriptor)
        if pre_fit[0]<0 and random.random()<fix_rate:
            fixed_ind=encoder(greedy_fix(decoder(ind, knap_instance_descriptor), knap_instance_descriptor),knap_instance_descriptor) 
        else:
            fixed_ind=ind.copy()
        lower_knap_instance_descriptor = adjust_allowed_items(decoder(fixed_ind, knap_instance_descriptor), load_file_instance(file_path))
        lower_knap_solution=Genetic_lower_knap(lower_knap_instance_descriptor, swarmsize_lower, iterations_lower, zeros_initial_proportion, prob_mutation, it_fix_lower, fix_rate)
        fitness=fitness_check_upper_knapsack(decoder(fixed_ind, knap_instance_descriptor), lower_knap_solution[1], knap_instance_descriptor)
        fit_list.append(fitness)
        first_gen_fixed.append(fixed_ind)
        
        lower_ind_list.append(lower_knap_solution[0])
        
    new_gen=first_gen_fixed.copy()
    new_fit_list=fit_list.copy()

    for it in range(iterations):
        fix= it%it_fix==0
        hijos=cruce_1P(new_gen, prob_mutation)
        new_gen, new_fit_list=selection_elitism_upper_knap(new_gen, hijos, new_fit_list, knap_instance_descriptor, file_path,  swarmsize_lower, iterations_lower, zeros_initial_proportion, prob_mutation, it_fix_lower, fix, fix_rate)
        
        best_inds.append(new_gen[0])
        best_fits.append(new_fit_list[0])

    best_ind=best_inds[-1]
    best_fit=best_fits[-1]
    
    return best_ind, best_fit, best_inds, best_fits

In [9]:
#ALGORITMOS DE PROGRAMACION DINÁMICA PARA OBTENER VALOR Y PESO ÓPTIMOS DE LA INSTANCIA
def separate_values(biknap_instance_descriptor, n_mochila=1):
    N=len(biknap_instance_descriptor[2])
    if n_mochila==1:
        W=biknap_instance_descriptor[0]
    elif n_mochila==2:
        W=biknap_instance_descriptor[1]

    val=[]
    wt=[]
    for item in biknap_instance_descriptor[2]:
        val.append(item[2])
        wt.append(item[1])

    return N, W, val, wt

#función que por programacion dinámica calcula el peso de la solución óptima y los itmes que la conforman
def mochila_DP_peso(biknap_instance_descriptor, n_mochila=1):
    N, W, val, wt = separate_values(biknap_instance_descriptor,n_mochila)
    DP = [[] for _ in range(N+1)] 
    
    for  i in range(N + 1):
      DP[i] = np.zeros(W + 1);
    
    t1=time.time()
    
    for i in range(N):
      for j in range(W):
        if (i == 0 or j == 0):
          DP[i][0] = 0;
        elif (wt[i - 1] <= j): #se elige el item actual de la colección
          A = val[i - 1] + DP[i - 1][j - wt[i - 1]];
          B = DP[i - 1][j];
          #se comparan los dos valores y se elige el de mayor peso
          if (A > B):
            DP[i][j] = A;
          else:
            DP[i][j] = B;
          
        else:  #o se deja el item actual de la colección
            DP[i][j] = DP[i - 1][j];
    
    solucion_w=0
    for i in reversed(range(W+1)):
        if DP[-1][i] <=W:
            solucion_w=i
            break;

    items_solution=[]    #recuperar los items de la solución
    i=N-1
    j=W-1
    while(i>0 and j>0):
        if(DP[i][j] != DP[i-1][j]):
            items_solution.append(biknap_instance_descriptor[2][i-1])
            j-=items_solution[-1][1]
        i-=1
    
    t2=time.time()
    t=t2-t1

    return solucion_w, items_solution, t

#función que por programacion dinámica calcula el valor de la solución óptima
def mochila_DP_valor(biknap_instance_descriptor, n_mochila=1):

    N, W, val, wt = separate_values(biknap_instance_descriptor, n_mochila)
    
    mat = [[] for _ in range(N+1)] 
    for  i in range(N + 1):
      mat[i] = [[] for _ in range(W + 1)]
        
    for r in range( W + 1):
        mat[0][r] = 0
    for c in range (N + 1):
        mat[c][0] = 0

    t1=time.time()
    for item in range(1,N):
        for capacity in range(1,W):
            maxValWithoutCurr = mat[item - 1][capacity]; 
            maxValWithCurr = 0; 
    
            weightOfCurr = wt[item - 1]; 
            if (capacity >= weightOfCurr): #Si el item actual coge en la mochila
                maxValWithCurr = val[item - 1]; 
                remainingCapacity = capacity - weightOfCurr; 
                maxValWithCurr += mat[item - 1][remainingCapacity]; # Se añade el máximo valor obtenible con la capacidad restante 
            
    
            mat[item][capacity] = max(maxValWithoutCurr, maxValWithCurr); # Se escoge el mayor de los dos valores

    t2=time.time()
    t=t2-t1
    return mat[N-1][W-1], t


#función que dado un conjunto de iterms calcula su valor y peso total
def calc_peso_valor_solucion(solution_item_collection):  
    peso_sum=0
    valor_sum=0
    for item in solution_item_collection:
        peso_sum+=item[1]
        valor_sum+=item[2]
    return peso_sum, valor_sum

#función que calcula la solución óptima mediante programación dinámica en problemas de mochila binivel en que los tamaños de ambas mochilas son múltiplos el uno del otro (la mitad, iguales, el doble)
def bilevel_knapsack_dynamic_programming_solution(filename): 
    biKnap_instance_descriptor=load_file_instance(filename)
    peso_mochila1=biKnap_instance_descriptor[0]
    peso_mochila2=biKnap_instance_descriptor[1]
    
    if(peso_mochila1<=peso_mochila2):    #si la mochila superior es igual o más pequeña que la inferior, se calcula lo óptimo para la primera mochila y luego lo óptimo en con la inferior para los items restantes
        solucion=mochila_DP_peso(biKnap_instance_descriptor)
        lower_biKnap_instance_descriptor=adjust_allowed_items(solucion[1], biKnap_instance_descriptor)
        lower_solucion=mochila_DP_peso(lower_biKnap_instance_descriptor, 2)
        
        peso_sol_opt_upper, valor_sol_opt_upper=calc_peso_valor_solucion(solucion[1])
        peso_sol_opt_lower, valor_sol_opt_lower=calc_peso_valor_solucion(lower_solucion[1])
        
    else:  #si la mochila superior es más grande (el doble), se calcula tres veces con el tamaño de la inferior, quitando items cada vez, simulando las dos primeras los items en la mochila superior
        solucion=mochila_DP_peso(biKnap_instance_descriptor, 2)
        biKnap_instance_descriptor=load_file_instance(filename)
        biKnap_instance_descriptor_2=adjust_allowed_items(solucion[1], biKnap_instance_descriptor)
        solucion2=mochila_DP_peso(biKnap_instance_descriptor_2, 2)
        biKnap_instance_descriptor=load_file_instance(filename)
        lower_biKnap_instance_descriptor=adjust_allowed_items(solucion2[1], biKnap_instance_descriptor_2)
        lower_solucion=mochila_DP_peso(lower_biKnap_instance_descriptor, 2)
        
        peso_sol_opt_lower, valor_sol_opt_lower=calc_peso_valor_solucion(lower_solucion[1])
        peso_sol_opt_upper1, valor_sol_opt_upper1=calc_peso_valor_solucion(solucion[1])
        peso_sol_opt_upper2, valor_sol_opt_upper2=calc_peso_valor_solucion(solucion2[1])
        peso_sol_opt_upper=peso_sol_opt_upper1+peso_sol_opt_upper2
        valor_sol_opt_upper=valor_sol_opt_upper1+valor_sol_opt_upper2

    return [peso_sol_opt_upper, valor_sol_opt_upper, solucion[1]], [peso_sol_opt_lower, valor_sol_opt_lower, lower_solucion[1]]



In [11]:
#Funciones para ejecución del algoritmo genético binivel con segundo nivel por porgramacion dinámica (control de la varianza)
def Genetic_bilevelKnapsack_2lvl_bruteforce(file_path, swarmsize=40, iterations=50, zeros_initial_proportion=0.5, prob_mutation=0.01, it_fix=10, fix_rate=0.5):
    
    fit_list=[]
    first_gen_fixed=[]
    lower_ind_list=[]
    best_inds=[]
    best_fits=[]

    knap_instance_descriptor=load_file_instance(file_path)
    random_enc_pop=random_x(len(knap_instance_descriptor[2]), swarmsize, zeros_initial_proportion) 
    
    for ind in random_enc_pop:
        pre_fit=fitness_check_knapsack(decoder(ind, knap_instance_descriptor), knap_instance_descriptor)
        if pre_fit[0]<0 and random.random()<fix_rate:
            fixed_ind=encoder(greedy_fix(decoder(ind, knap_instance_descriptor), knap_instance_descriptor),knap_instance_descriptor) 
        else:
            fixed_ind=ind.copy()
        
        lower_knap_instance_descriptor = adjust_allowed_items(decoder(fixed_ind, knap_instance_descriptor), load_file_instance(file_path))
        #lower_knap_solution=Genetic_lower_knap(lower_knap_instance_descriptor, 40, 50, zeros_initial_proportion, prob_mutation, it_fix_lower)
        lower_DP_solution=mochila_DP_peso(lower_knap_instance_descriptor, 2)
        lower_DP_solution_values=calc_peso_valor_solucion(lower_DP_solution[1])
        ogFormat_lower_sol0=encoder(lower_DP_solution[1], knap_instance_descriptor)
        ogFormat_lower_sol1=[lower_knap_instance_descriptor[0]-lower_DP_solution_values[0], lower_DP_solution_values[1]]
        lower_knap_solution=[ogFormat_lower_sol0,ogFormat_lower_sol1]
        
        fitness=fitness_check_upper_knapsack(decoder(fixed_ind, knap_instance_descriptor), lower_knap_solution[1], knap_instance_descriptor)
        fit_list.append(fitness)
        first_gen_fixed.append(fixed_ind)
        
        lower_ind_list.append(lower_knap_solution[0])
        
    new_gen=first_gen_fixed.copy()
    new_fit_list=fit_list.copy()

    for it in range(iterations):
        fix= it%it_fix==0
        hijos=cruce_1P(new_gen, prob_mutation)
        new_gen, new_fit_list=selection_elitism_upper_knap_2lvl_bruteforce(new_gen, hijos, new_fit_list, knap_instance_descriptor, file_path, zeros_initial_proportion, prob_mutation, fix, fix_rate)
        
        best_inds.append(new_gen[0])
        best_fits.append(new_fit_list[0])

    best_ind=best_inds[-1]
    best_fit=best_fits[-1]
    
    return best_ind, best_fit, best_inds, best_fits


def selection_elitism_upper_knap_2lvl_bruteforce(x_pop, x_hijos, fit_list, knap_instance_descriptor, file_path, zeros_initial_proportion=0.5, prob_mutation=0.01, fix=False, fix_rate=0.5, debug_check=False):   #función de selección elitista para el primer nivel
    new_gen=[]
    new_fit_list=[]
    x_pop_sorted=[]
    tam_pop=len(x_pop)
    
    for x_hijo in x_hijos:    #para cada hijo se calcula su fitness
        x_hijo_items=decoder(x_hijo, knap_instance_descriptor)
        lower_knap_instance_descriptor = adjust_allowed_items(x_hijo_items, load_file_instance(file_path))
        #lower_knap_solution=Genetic_lower_knap(lower_knap_instance_descriptor, swarmsize_lower, iterations_lower, zeros_initial_proportion, prob_mutation, it_fix_lower)
        lower_DP_solution=mochila_DP_peso(lower_knap_instance_descriptor, 2)
        lower_DP_solution_values=calc_peso_valor_solucion(lower_DP_solution[1])
        ogFormat_lower_sol0=encoder(lower_DP_solution[1], knap_instance_descriptor)
        ogFormat_lower_sol1=[lower_knap_instance_descriptor[0]-lower_DP_solution_values[0], lower_DP_solution_values[1]]
        lower_knap_solution=[ogFormat_lower_sol0,ogFormat_lower_sol1]
        
        fitness=fitness_check_upper_knapsack(x_hijo_items, lower_knap_solution[1], knap_instance_descriptor)
        if fix:
            if fitness[0]<0 and random.random()<fix_rate:
                x_hijo_items_fixed=greedy_fix(x_hijo_items, knap_instance_descriptor)
                x_hijo_fixed=encoder(x_hijo_items_fixed, knap_instance_descriptor)
                lower_knap_instance_descriptor = adjust_allowed_items(x_hijo_items_fixed, load_file_instance(file_path))
                #lower_knap_solution=Genetic_lower_knap(lower_knap_instance_descriptor, swarmsize_lower, iterations_lower, zeros_initial_proportion, prob_mutation, it_fix_lower)
                lower_DP_solution=mochila_DP_peso(lower_knap_instance_descriptor, 2)
                lower_DP_solution_values=calc_peso_valor_solucion(lower_DP_solution[1])
                ogFormat_lower_sol0=encoder(lower_DP_solution[1], knap_instance_descriptor)
                ogFormat_lower_sol1=[lower_knap_instance_descriptor[0]-lower_DP_solution_values[0], lower_DP_solution_values[1]]
                lower_knap_solution=[ogFormat_lower_sol0,ogFormat_lower_sol1]
                
                fitness=fitness_check_upper_knapsack(x_hijo_items_fixed, lower_knap_solution[1], knap_instance_descriptor)
                x_pop.append(x_hijo_fixed)
            else:
                x_pop.append(x_hijo)
        else:
            x_pop.append(x_hijo)
            
        fit_list.append(fitness)
    

    fit_list_sorted=sort_fitness_upper_knap(fit_list, debug_check)    #se ordenan los fitness de padres e hijos
    for fit_i in fit_list_sorted:
        x_pop_sorted.append(x_pop[fit_i[-1]])     #se ordenan los individuios de acuredo al orden del fitness

    for i in range(len(x_pop_sorted)):
        x_pop_sorted[i]=[x_pop_sorted[i], [fit_list_sorted[i][0], fit_list_sorted[i][1], fit_list_sorted[i][2],  fit_list_sorted[i][3]]]  
    
    if(debug_check):
        print("selection:", tam_pop, len(fit_list), len (fit_list_sorted), len(x_pop_sorted))
    
    for i in range(tam_pop):   #los individuos con el mayor fitness sobreviven
       new_gen.append(x_pop_sorted[i][0])
       new_fit_list.append(x_pop_sorted[i][1])
        
    return new_gen, new_fit_list

In [12]:
#FUNCION PARA EJECUCIONES EN LOTE DE INSTANCIAS DE LA MOCHILA BINIVEL
def pruebas_bilevelKnapsack_ag(n, file_names, full_solutions_filename, partial_results_filename, n_pruebas=10, par_comb=[[40, 50], [40, 50], [0.95, 5, 10, 1]], verbose=True):
    resultados_totales=[]
    full_solutions=load_file_instance(full_solutions_filename)
    
    for h in par_comb:     #para cada par de tamaños de iteraciones del ag inferior
        resultados_totales.append([[h[1][0], h[1][1]]])
        if(verbose):
            print("Pop: ", h[1][0], "- Iteraciones: ", h[1][1])
        t1_global=time.time()

        for i in range(len(file_names)):   #se ejecutan n_pruebas sobre cada una de las 60 instanicas
            resultados_instancia=[]
            
            file_path=file_names[i]       #carga de la instancia y su solución por programación dinámica
            biKnap_instance_descriptor=load_file_instance(file_path)
            full_solution_DP=[biKnap_instance_descriptor[0]-full_solutions[i][1][0][0], full_solutions[i][1][0][1], biKnap_instance_descriptor[1]-full_solutions[i][1][1][0], full_solutions[i][1][1][1]]
            full_solution_lower_solution_value=full_solutions[i][1][1][1]

            if(verbose):
                print(i, file_path, " - ", full_solution_DP)
                
            resultados_instancia.append(i)
            resultados_instancia.append(file_path)
            tiempo_total=0
            resultados_validos_totales=0
            better_than_opt_totales=0

    
            for j in range(n_pruebas):     #prueba unitaria para una instancia (se ejecuta n_pruebas veces)
                better_than_opt=0
                
                t1=time.time()
                result_ag=Genetic_bilevelKnapsack(file_path, swarmsize=h[0][0], iterations=h[0][1], swarmsize_lower=h[1][0], iterations_lower=h[1][1], zeros_initial_proportion=h[2][0], it_fix=h[2][1], it_fix_lower=h[2][2], fix_rate=h[2][3])
                t2=time.time()
                t=t2-t1
                
                tiempo_total+=t
            
                #comprobación de la diferencia entre la solución óptima de la instnaica y el valor obtenido por el ag expresada en porcentaje sobre el valor de la solución óptima de la instancia
                valid_solution=(full_solution_lower_solution_value-result_ag[1][3])/full_solution_lower_solution_value
                if (valid_solution>=0):
                    resultados_validos_totales+=valid_solution
                else: #si por alguún motivo son mejores los resultados del algortimo genético que los de programación dinámica, se registra
                    resultados_validos_totales+= (-valid_solution)
                    better_than_opt=1
                better_than_opt_totales+=better_than_opt
                
                resultados_instancia.append([j, result_ag[0], result_ag[1], abs(valid_solution), better_than_opt, t])

                if(verbose):
                    print("- ", resultados_instancia[-1][0], resultados_instancia[-1][2], resultados_instancia[-1][3], resultados_instancia[-1][4], resultados_instancia[-1][5])

            
            resultados_validos_media_global_instancia=resultados_validos_totales/n_pruebas      #estadísticas del conjunto de pruebas de la instncia
            tiempo_medio=tiempo_total/n_pruebas
            
            resultados_instancia.append(resultados_validos_media_global_instancia)
            resultados_instancia.append(better_than_opt_totales)
            resultados_instancia.append(tiempo_medio)
            resultados_instancia.append(tiempo_total)
    
            t2_global=time.time()
            t_global=t2_global-t1_global
    
            resultados_totales[-1].append(resultados_instancia)
            
            if(verbose):       #resumen de las ejecuciones de la instancia
                print(i+1, "/", len(file_names), " --- ", resultados_validos_media_global_instancia, " --- ", better_than_opt_totales, " --- ", tiempo_medio, " --- ", t_global, '\n')
    
        save_instance_file(resultados_totales, partial_results_filename)
        
    return resultados_totales


#Para pruebas con el algoritmo genético con segundo nivel con programación dinámica (sin varianza para control)
def pruebas_bilevelKnapsack_2lvl_bruteforce(n, file_names, full_solutions_filename, partial_results_filename, n_pruebas=10, par_comb=[[40, 50], [0, 0], [0.95, 5, 10, 1]], verbose=True):
    resultados_totales=[]
    full_solutions=load_file_instance(full_solutions_filename)
    
    for h in par_comb:     #para cada par de tamaños de iteraciones del ag inferior
        resultados_totales.append([[h[1][0], h[1][1]]])
        if(verbose):
            print("Pop: ", h[1][0], "- Iteraciones: ", h[1][1], " (Segundo nivel por programación dinámica)")
        t1_global=time.time()

        for i in range(len(file_names)):   #se ejecutan n_pruebas sobre cada una de las 60 instanicas
            resultados_instancia=[]
            
            file_path=file_names[i]       #carga de la instancia y su solución por programación dinámica
            biKnap_instance_descriptor=load_file_instance(file_path)
            full_solution_DP=[biKnap_instance_descriptor[0]-full_solutions[i][1][0][0], full_solutions[i][1][0][1], biKnap_instance_descriptor[1]-full_solutions[i][1][1][0], full_solutions[i][1][1][1]]
            full_solution_lower_solution_value=full_solutions[i][1][1][1]

            if(verbose):
                print(i, file_path, " - ", full_solution_DP)
                
            resultados_instancia.append(i)
            resultados_instancia.append(file_path)
            tiempo_total=0
            resultados_validos_totales=0
            better_than_opt_totales=0

    
            for j in range(n_pruebas):     #prueba unitaria para una instancia (se ejecuta n_pruebas veces)
                better_than_opt=0
                
                t1=time.time()
                result_ag=Genetic_bilevelKnapsack_2lvl_bruteforce(file_path, swarmsize=h[0][0], iterations=h[0][1], zeros_initial_proportion=h[2][0], it_fix=h[2][1], fix_rate=h[2][3])
                t2=time.time()
                t=t2-t1
                
                tiempo_total+=t
            
                #comprobación de la diferencia entre la solución óptima de la instnaica y el valor obtenido por el ag expresada en porcentaje sobre el valor de la solución óptima de la instancia
                valid_solution=(full_solution_lower_solution_value-result_ag[1][3])/full_solution_lower_solution_value
                if (valid_solution>=0):
                    resultados_validos_totales+=valid_solution
                else: #si por alguún motivo son mejores los resultados del algortimo genético que los de programación dinámica, se registra
                    resultados_validos_totales+= (-valid_solution)
                    better_than_opt=1
                better_than_opt_totales+=better_than_opt
                
                resultados_instancia.append([j, result_ag[0], result_ag[1], abs(valid_solution), better_than_opt, t])

                if(verbose):
                    print("- ", resultados_instancia[-1][0], resultados_instancia[-1][2], resultados_instancia[-1][3], resultados_instancia[-1][4], resultados_instancia[-1][5])

            
            resultados_validos_media_global_instancia=resultados_validos_totales/n_pruebas      #estadísticas del conjunto de pruebas de la instncia
            tiempo_medio=tiempo_total/n_pruebas
            
            resultados_instancia.append(resultados_validos_media_global_instancia)
            resultados_instancia.append(better_than_opt_totales)
            resultados_instancia.append(tiempo_medio)
            resultados_instancia.append(tiempo_total)
    
            t2_global=time.time()
            t_global=t2_global-t1_global
    
            resultados_totales[-1].append(resultados_instancia)
            
            if(verbose):       #resumen de las ejecuciones de la instancia
                print(i+1, "/", len(file_names), " --- ", resultados_validos_media_global_instancia, " --- ", better_than_opt_totales, " --- ", tiempo_medio, " --- ", t_global, '\n')
    
        save_instance_file(resultados_totales, partial_results_filename)
        
    return resultados_totales